<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/y_Ax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp

wp.init()
device = "cuda"

Warp version 1.12.0 is ready!


# Without TILE


In [ ]:
import warp as wp
import numpy as np
import time
wp.init()

@wp.kernel
def mat_vec_naive(A: wp.array2d(dtype=wp.float32), # Explicitly float32
                  x: wp.array(dtype=wp.float32),
                  y: wp.array(dtype=wp.float32),
                  rows: int,
                  cols: int):

    i = wp.tid()
    if i >= rows:
        return

    row_sum = wp.float32(0.0)

    for j in range(cols):
        row_sum += A[i, j] * x[j]

    y[i] = row_sum

def main():
  M, N = 2048, 2048
  device = "cuda"

  A_np = np.random.rand(M, N).astype(np.float32)
  x_np = np.random.rand(N).astype(np.float32)

  A_wp = wp.from_numpy(A_np, dtype=wp.float32, device=device)
  x_wp = wp.from_numpy(x_np, dtype=wp.float32, device=device)
  y_wp = wp.zeros(M, dtype=wp.float32, device=device)

  print(f"Launching Fixed Non-Tiled Mat-Vec on {device}...")

  start = time.time()
  wp.launch(
      kernel=mat_vec_naive,
      dim=M,
      inputs=[A_wp, x_wp, y_wp, M, N],
      device=device
  )

  wp.synchronize()
  duration = (time.time() - start) * 1000

  # Verify
  y_final = y_wp.numpy()
  y_expected = np.dot(A_np, x_np)
  max_diff = np.max(np.abs(y_final - y_expected))

  print("-" * 30)
  print(f"Execution Time: {duration:.2f} ms")
  print(f"Max Absolute Error: {max_diff:.2e}")
  print("-" * 30)

if __name__ == "__main__":
  main()

Launching Fixed Non-Tiled Mat-Vec on cuda...
Module __main__ 06c43e8 load on device 'cuda:0' took 1335.51 ms  (compiled)
------------------------------
Execution Time: 1338.84 ms
Max Absolute Error: 1.16e-03
------------------------------


# With blocks

In [ ]:
import warp as wp
import numpy as np
import time

wp.init()

# --- 1. The Tiled Kernel ---
# Tile size (number of threads helping per row)
# 128 is a "sweet spot" for occupancy on RTX 6000 / Ampere cards
TPB = 128

@wp.kernel
def mat_vec_tiled(A: wp.array2d(dtype=wp.float32),
                  x: wp.array(dtype=wp.float32),
                  y: wp.array(dtype=wp.float32),
                  rows: int,
                  cols: int):

    # 2D Indexing: tile_id is the row, local_id is the thread's rank in the team
    tile_id, local_id = wp.tid()

    if tile_id >= rows:
        return

    # Use explicit float32 constructor for the partial sum
    partial_sum = wp.float32(0.0)

    # Teamwork: Threads jump by TPB to process the row in parallel
    # This ensures that thread 0, 1, 2... access A[tile_id, 0, 1, 2...] together
    for j in range(local_id, cols, TPB):
        partial_sum += A[tile_id, j] * x[j]

    # Combine partial sums safely from the 128 threads into the final row result
    wp.atomic_add(y, tile_id, partial_sum)

#Launch
M, N = 2048, 2048
device = "cuda"

# A. Force NumPy to float32
A_np = np.random.rand(M, N).astype(np.float32)
x_np = np.random.rand(N).astype(np.float32)

# B. Force Warp to use float32
A_wp = wp.from_numpy(A_np, dtype=wp.float32, device=device)
x_wp = wp.from_numpy(x_np, dtype=wp.float32, device=device)
y_wp = wp.zeros(M, dtype=wp.float32, device=device)

print(f"Launching Tiled Mat-Vec on {device}...")

start = time.time()

# C. Launch with 2D dimensions: (Number of rows, Threads per row)
wp.launch(
    kernel=mat_vec_tiled,
    dim=(M, TPB),
    inputs=[A_wp, x_wp, y_wp, M, N],
    device=device
)

wp.synchronize()
duration = (time.time() - start) * 1000

# D. Verify
y_final = y_wp.numpy()
y_expected = np.dot(A_np, x_np)
max_diff = np.max(np.abs(y_final - y_expected))

print("-" * 30)
print(f"Execution Time: {duration:.2f} ms")
print(f"Max Absolute Error: {max_diff:.2e}")
print("-" * 30)


ModuleNotFoundError: No module named 'warp'

## Tile code.. Efficient.

In [19]:
import warp as wp
import numpy as np

wp.config.verify_cuda = True
wp.init()

# TPB = Threads Per Block / Tile Size
# We match this to the width of the matrix (or a factor of it)
TPB = 128

@wp.kernel
def mat_vec_tiled_kernel(A: wp.array2d(dtype=wp.float32),
                         x: wp.array(dtype=wp.float32),
                         y: wp.array(dtype=wp.float32)):

    # Each Tile handles exactly one row of matrix A
    row_idx = wp.tid()

    # 1. Load x as a 2D tile (1 row, TPB columns) to match tile_A_row
    tile_x = wp.tile_load(x, shape=(TPB,), offset=(0,))

    # 2. Load A row as a 2D Tile (1 row, TPB columns)
    tile_A_2d = wp.tile_load(A, shape=(1, TPB), offset=(row_idx, 0))

    tile_A_1d = wp.tile_reshape(tile_A_2d, shape=(TPB,))

    # 3. Compute (A_row * x)
    # Now both are 2D tiles of shape (1, TPB), so math is allowed element-wise
    #tile_prod = tile_A_1d * tile_x
    tile_prod = tile_x

    # 4. Reduce
    # tile_sum on a 2D tile returns a 1x1 tile
    row_sum_tile = wp.tile_sum(tile_prod)

    # 5. Store
    y[row_idx] = row_sum_tile[0] # Use 2D indexing for the 1x1 result

# --- Execution Logic ---
device = "cuda"
ROWS = 128
COLS = 128 # Must match TPB for this simple example

# Initialize data
A_np = np.random.rand(ROWS, COLS).astype(np.float32)
x_np = np.random.rand(COLS).astype(np.float32)
print(x_np, '\n', np.sum(x_np))
A_wp = wp.array2d(A_np, device=device)
x_wp = wp.array(x_np, device=device)
y_wp = wp.zeros(ROWS, dtype=wp.float32, device=device)

# Launch: One thread-block (tile) per ROW
wp.launch(
    kernel=mat_vec_tiled_kernel,
    dim=ROWS,
    inputs=[A_wp, x_wp, y_wp],
    device=device
)

# Verification
y_final = y_wp.numpy()
expected = A_np @ x_np
expected =  x_np

print(y_final)
print(x_np)

print(f"--- Performance Report: y=Ax ---")
#print(f"Max Difference: {np.mean(np.abs(y_final - expected)):.2e}")
#print(f"Max Difference: {np.max(np.abs(y_final - expected)):.2e}")

[0.26911953 0.663009   0.8177598  0.24323303 0.7930256  0.5385127
 0.6590259  0.36412773 0.83148265 0.868851   0.5728996  0.58222896
 0.72787416 0.47168928 0.4636952  0.5165661  0.27578816 0.57627803
 0.72778547 0.4479102  0.1308917  0.28451797 0.81633264 0.19063559
 0.09142798 0.5068254  0.15551704 0.9054996  0.6329682  0.45344862
 0.1825451  0.19117928 0.8049255  0.5376274  0.5579803  0.13483717
 0.31729886 0.9499202  0.7412107  0.04702318 0.07508089 0.20742491
 0.27972335 0.61607337 0.6941674  0.6986386  0.4624867  0.12753029
 0.9725622  0.29266012 0.6096283  0.5517062  0.7514565  0.5487246
 0.2283376  0.16422538 0.50814885 0.4379956  0.14466898 0.10267304
 0.7502517  0.19437683 0.01118999 0.00873806 0.9245078  0.58823246
 0.68381155 0.16879024 0.53852767 0.25509104 0.58150154 0.98104656
 0.02229636 0.2664455  0.09430146 0.9486096  0.01610925 0.6071222
 0.2185083  0.12195847 0.32797348 0.30159122 0.8680654  0.21493271
 0.03843064 0.69240516 0.4902822  0.08006536 0.40094686 0.3138760

# Mat Mult C=A*B


In [ ]:
import numpy as np
import warp as wp

wp.init()

@wp.kernel
def print_matrix_v3(data: wp.array(dtype=wp.float32, ndim=2),
                    rows: int,
                    cols: int):

    # This is the most compatible way to get 2D indices in Warp
    # It explicitly tells the compiler to expect a 2-element vector
    i, j = wp.tid()

    if i < rows and j < cols:
        val = data[i, j]
        wp.printf("Row: %d, Col: %d, Val: %f\n", i, j, val)

# Small test case
data_np =np.random.rand(3, 3).astype(np.float32) # 3x3 Identity matrix
#data_wp = wp.from_numpy(data_np, device="cuda")
data_wp = wp.array(data_np, dtype=wp.float32, device="cuda")


print("--- Launching ---")
wp.launch(kernel=print_matrix_v3,
          dim=(3, 3), # The tuple here MUST match the i, j unpacking above
          inputs=[data_wp, 3, 3],
          device="cuda")

wp.synchronize()

--- Launching ---


In [ ]:
import numpy as np
import warp as wp

wp.init()

@wp.kernel
def matmul_2d_kernel(A: wp.array(dtype=wp.float32, ndim=2),
                     B: wp.array(dtype=wp.float32, ndim=2),
                     C: wp.array(dtype=wp.float32, ndim=2),
                     M: int, N: int, K: int):

    # Correct way to handle 2D thread indices
    i, j = wp.tid()

    if i < M and j < N:
        tmp = float(0.0)
        for k in range(K):
            # Explicitly indexing into the 2D arrays
            tmp += A[i, k] * B[k, j]

        C[i, j] = tmp

# Setup dimensions: A (M x K), B (K x N) -> C (M x N)
M, K, N = 512, 256, 512

# Initialize arrays on GPU
A_np = np.random.rand(M, K).astype(np.float32)
B_np = np.random.rand(K, N).astype(np.float32)

A_wp = wp.array(A_np, dtype=wp.float32, device="cuda")
B_wp = wp.array(B_np, dtype=wp.float32, device="cuda")
C_wp = wp.zeros(shape=(M, N), dtype=wp.float32, device="cuda")

# Launch using 2D dimensions
# Warp automatically assigns tid.x to M and tid.y to N
wp.launch(kernel=matmul_2d_kernel,
          dim=(M, N),
          inputs=[A_wp, B_wp, C_wp, M, N, K],
          device="cuda")

wp.synchronize()
print("Matrix multiplication complete.")
print(f"Result Shape: {C_wp.shape}")

Module __main__ 485badd load on device 'cuda:0' took 520.40 ms  (compiled)
Matrix multiplication complete.
Result Shape: (512, 512)


## With tiles.

In [ ]:
import warp as wp
import numpy as np

wp.init()

TILE = 16

# ---------------------------------------------------
# GPU Kernel
# ---------------------------------------------------
@wp.kernel
def matmul_tiled(
    A: wp.array2d(dtype=wp.float32),
    B: wp.array2d(dtype=wp.float32),
    C: wp.array2d(dtype=wp.float32),
    N: int
):
    i, j = wp.tid()  # thread indices

    # Shared memory tiles
    tileA = wp.shared_array(shape=(TILE, TILE), dtype=wp.float32)
    tileB = wp.shared_array(shape=(TILE, TILE), dtype=wp.float32)

    sum = float(0.0)

    for t in range(N // TILE):

        tileA[i % TILE, j % TILE] = A[i, t * TILE + j % TILE]
        tileB[i % TILE, j % TILE] = B[t * TILE + i % TILE, j]

        wp.syncthreads()

        for k in range(TILE):
            sum += tileA[i % TILE, k] * tileB[k, j % TILE]

        wp.syncthreads()

    C[i, j] = sum


# ---------------------------------------------------
# Host Code
# ---------------------------------------------------
N = 512

A_np = np.random.rand(N, N).astype(np.float32)
B_np = np.random.rand(N, N).astype(np.float32)
C_np = np.zeros((N, N), dtype=np.float32)

A = wp.array(A_np, dtype=wp.float32, device="cuda")
B = wp.array(B_np, dtype=wp.float32, device="cuda")
C = wp.array(C_np, dtype=wp.float32, device="cuda")

wp.launch(
    kernel=matmul_tiled,
    dim=(N, N),
    inputs=[A, B, C, N],
    device="cuda"
)

C_result = C.numpy()

print("C[0,0] =", C_result[0,0])

ModuleNotFoundError: No module named 'warp'

In [ ]:
import warp as wp
import numpy as np
import torch

# 1. Initialize with verification to catch Colab crashes before they happen
wp.config.verify_cuda = True
wp.init()

# Define tile size (Must be a power of 2 for efficiency)
# A 16x16 tile of floats uses only 1KB of shared memory (Safe for Colab)
TILE_DIM = 16

@wp.kernel
def tiled_shared_sum_kernel(
    input_data: wp.array2d(dtype=float),
    output_sums: wp.array(dtype=float)
):
    # i, j are the tile indices
    i, j = wp.tid()

    # --- SHARED MEMORY STEP ---
    # wp.tile_load automatically allocates and loads data into
    # the GPU's fast Shared Memory (L1 Cache equivalent)
    local_tile = wp.tile_load(input_data, shape=(TILE_DIM, TILE_DIM), offset=(i*TILE_DIM, j*TILE_DIM))

    # Perform math purely in shared memory (Blazing fast)
    # Extract the scalar value by indexing the single-element tile returned by wp.tile_sum
    tile_total = wp.tile_sum(local_tile)[0]

    # Write back to global memory
    # We use atomic_add because multiple tiles might contribute to a global diagnostic
    wp.atomic_add(output_sums, 0, tile_total)

# --- Execution Block ---

device = "cuda"
N = 128 # Grid size

# Create dummy velocity energy data
data_np = np.random.rand(N, N).astype(np.float32)
data_wp = wp.array(data_np, device=device)
sum_out = wp.zeros(1, dtype=float, device=device)

# Launch using the TILED launch command
# dim represents the number of TILES, not the number of threads
wp.launch(
    kernel=tiled_shared_sum_kernel,
    dim=(N // TILE_DIM, N // TILE_DIM),
    inputs=[data_wp, sum_out],
    device=device
)

print(f"Total Sum via Shared Memory: {sum_out.numpy()[0]:.4f}")
print(f"Verification (NumPy): {np.sum(data_np):.4f}")

Total Sum via Shared Memory: 1846.2889
Verification (NumPy): 8184.7861
